# record

> Recording model turns for deterministic replay.

`CachedChat` records the first result for an ask and replays it later. Tests avoid repeated model cost, latency, and network failures. `URAI_RECORD_CHAT` controls re-recording.

::: {.callout-note}
This testing utility can move to a separate package. `diskcache` backs it.
:::

In [ ]:
#| default_exp record

In [ ]:
#| export
import json, os, re
from hashlib import sha256
from fastcore.all import L, Path, store_attr, ifnone, listify, str2bool
from urai.core import Resp, UsageStats, resp_text
from urai.msgs import is_media, tool_rows
from urai.chat import Chat

In [ ]:
#| hide
import tempfile
from fastcore.test import test_eq, test_fail
from urai.opts import RUNTIMES, ChatOpts, Runtime, register_runtime
from urai.loop import ToolLoopMixin

## What is worth remembering

A backend rejection belongs to the ask and is recorded. Rate limits and transport failures are transient. They are raised without entering the cache.

In [ ]:
#| export
class NotRecordable(TypeError):
    "A reply that cannot be stored as one. Never cached, so the next ask tries again."


In [ ]:
#| export
CHAT_CACHE = 'chatcache'   #: default diskcache directory
KEY_VERSION = 3            #: part of every key; bump to invalidate every recording everywhere

_transient_re = re.compile(
    r'rate.?limit|timed? ?out|timeout|temporarily|try again|50[234]\b|429', re.I)

def is_transient(e):
    """Is `e` a failure of the moment rather than of the ask? Those are not worth remembering.

    A reply that cannot be stored counts: caching it would replay the refusal instead of the
    answer, and the next ask would never get a chance to record one.
    """
    if isinstance(e, NotRecordable): return True
    return isinstance(e, (ConnectionError, TimeoutError, OSError)) or bool(_transient_re.search(str(e)))

In [ ]:
test_eq(is_transient(RuntimeError('429 rate limit')), True)
test_eq(is_transient(TimeoutError()), True)
test_eq(is_transient(ConnectionError('reset')), True)
test_eq(is_transient(RuntimeError('503 Service Unavailable')), True)
test_eq(is_transient(ValueError('unknown tool: frobnicate')), False)   # a fact about the ask

## The cache

`record` controls cache misses. When unset, it reads `URAI_RECORD_CHAT` as a boolean. The value `0` disables live recording.

In [ ]:
#| export
class RecordCache:
    "Record what a call returned the first time and replay it after. The primitive under `CachedChat`."
    def __init__(self,
                 path=None,     # the diskcache directory; None -> `CHAT_CACHE`
                 record=None,   # let a miss run for real; None -> read `env`
                 env='URAI_RECORD_CHAT',
                 version=None): # part of every key; None -> `KEY_VERSION`
        from diskcache import Cache
        self.cache, self.env = Cache(str(path or CHAT_CACHE)), env
        self.version = ifnone(version, KEY_VERSION)
        self.record = str2bool(os.getenv(env) or '') if record is None else record

    def key(self, *parts):
        "Stable hash of `parts`, with the cache version in it."
        return sha256(json.dumps([self.version, *parts], sort_keys=True,
                                 default=str).encode()).hexdigest()

    def __call__(self, key, f, what=''):
        "Replay `key`, else run `f()` and record what it did, including how it failed."
        if key in self.cache:
            got, val = self.cache[key]
            if got == 'exc': raise RuntimeError(val)
            return val
        if not self.record: raise KeyError(
            f'no recording for {what or key[:16]} - set {self.env}=1 and re-run to record it')
        try: val = f()
        except Exception as e:
            if not is_transient(e): self.cache[key] = ('exc', f'{type(e).__name__}: {e}')
            raise
        self.cache[key] = ('ok', val)
        return val

    def forget(self, key):
        "Drop one recording, for when what it captured was never the truth. Was there one?"
        return self.cache.pop(key, None) is not None

In [ ]:
d = tempfile.mkdtemp()
rec = RecordCache(d, record=True)
calls = []
test_eq(rec(rec.key('a'), lambda: calls.append(1) or 'first'), 'first')
test_eq(rec(rec.key('a'), lambda: calls.append(1) or 'second'), 'first')   # replayed
test_eq(len(calls), 1)

In [ ]:
test_eq(rec.key('a') == rec.key('a'), True)          # stable
test_eq(rec.key('a') == rec.key('b'), False)
test_eq(RecordCache(d, record=True, version=99).key('a') == rec.key('a'), False)  # version is in it

In [ ]:
# a failure about the ask is recorded and replays; a failure about the afternoon is not
test_fail(lambda: rec(rec.key('bad'), lambda: (_ for _ in ()).throw(ValueError('no such tool'))))
test_fail(lambda: rec(rec.key('bad'), lambda: 'never runs'), contains='ValueError: no such tool')

In [ ]:
test_fail(lambda: rec(rec.key('flaky'), lambda: (_ for _ in ()).throw(TimeoutError('429'))))
test_eq(rec.key('flaky') in rec.cache, False)        # forgotten, so the next run may try again

In [ ]:
# with recording off, a miss is an error that says how to fix itself
ro = RecordCache(d, record=False)
test_fail(lambda: ro(ro.key('unseen'), lambda: 'x'), contains='URAI_RECORD_CHAT=1')
test_eq(ro(rec.key('a'), lambda: 'x'), 'first')      # ...but a hit still replays

In [ ]:
test_eq(rec.forget(rec.key('a')), True)
test_eq(rec.forget(rec.key('a')), False)             # already gone

In [ ]:
os.environ['URAI_RECORD_CHAT'] = '0'
test_eq(RecordCache(d).record, False)                # '0' means no, not "a non-empty string"
os.environ['URAI_RECORD_CHAT'] = '1'
test_eq(RecordCache(d).record, True)
del os.environ['URAI_RECORD_CHAT']
test_eq(RecordCache(d).record, False)

## The chat

What goes into a key is everything a reply depends on: the model, how it was built, the system prompt, the tool names, and the conversation so far. Media is hashed rather than stored, so a key stays small.

In [ ]:
#| export
def as_resp(r):
    """One reply as something recordable.

    Most backends answer with a mapping. A local engine may answer with plain text, and `dict()`
    on a string raises a `ValueError` about sequence lengths rather than saying what is wrong.

    Anything else is refused rather than coerced. A streamed turn arrives here as a generator, and
    recording its `repr` would store a memory address as the answer and replay it for ever.
    """
    if isinstance(r, str): return {'role': 'assistant', 'content': r}
    try: return dict(r)
    except (TypeError, ValueError) as e:
        raise NotRecordable(f'cannot record a {type(r).__name__} as a reply') from e


def canon_msg(m):
    "Canonical cache-key fields for one history entry."
    media = [sha256(str(p).encode()).hexdigest()[:16] for p in listify(m.get('content')) if is_media(p)]
    tcs = tool_rows(m)
    return [m.get('role', ''), resp_text(m), media, tcs]

Most backends answer with a mapping. A local engine may answer with plain text, and `dict()` on a
string raises a `ValueError` about sequence lengths rather than saying what is wrong. That error
used to be recorded as the answer and replayed for ever.

Anything else is refused rather than coerced, and the refusal is counted as transient so it is never
cached. A streamed turn arrives here as a generator, and recording its `repr` would store a memory
address as the answer.

In [ ]:
as_resp('just text'), as_resp({'role': 'assistant', 'content': 'a mapping'})

In [ ]:
test_eq(as_resp('just text'), {'role': 'assistant', 'content': 'just text'})
test_eq(as_resp({'role': 'assistant', 'content': 'a mapping'}),
        {'role': 'assistant', 'content': 'a mapping'})
test_eq(as_resp(Resp({'role': 'assistant', 'content': 'a Resp'}))['content'], 'a Resp')
test_eq(resp_text(Resp(as_resp('just text'))), 'just text')

# a streamed turn arrives as a generator, and its `repr` is a memory address rather than an answer
def _gen(): yield 'chunk'
test_fail(lambda: as_resp(_gen()), contains='cannot record a generator')
test_fail(lambda: as_resp(42), contains='cannot record a int')
test_eq(is_transient(NotRecordable('x')), True)            # so it is never cached

In [ ]:
test_eq(canon_msg({'role': 'user', 'content': 'hi'}), ['user', 'hi', [], []])
from urai.msgs import ToolCall, mk_msg
test_eq(canon_msg({'role': 'assistant', 'content': '', 'tool_calls': [ToolCall('add', {'a': 1})]}),
        ['assistant', '', [], [('add', {'a': 1})]])

In [ ]:
png = b'\x89PNG\r\n\x1a\n' + b'\x00' * 40
m = canon_msg(mk_msg(['look', png]))
test_eq((m[0], m[1]), ('user', 'look'))
test_eq(len(m[2][0]), 16)              # the picture is hashed, not carried into the key

A recording stores the complete history added by one turn. For a tool turn, that includes the call, result, and final answer. Replaying only the answer would create a history the model never saw. It would also corrupt the cache key for the next turn.

In [ ]:
#| export
class CachedChat:
    "A `Chat` whose replies are recorded to disk and replayed on a second ask. A replay builds no engine."
    def __init__(self,
                 model=None,   # anything `Chat` takes; part of the key
                 path=None,    # the diskcache directory; None -> `CHAT_CACHE`
                 record=None,  # let a miss reach a real model; None -> `$URAI_RECORD_CHAT`
                 sp='',        # system prompt, part of the key
                 tools=None,   # tool *names* are part of the key; the real chat gets the tools
                 **kw):        # forwarded to `Chat` on a miss
        store_attr('model,sp,kw')
        self.tools, self.rec, self._chat, self.hist = L(tools), RecordCache(path, record), None, []
        self.use = UsageStats()   # folded from every reply, replayed or live, like `Chat.use`
        self._ctx = 0             # occupancy the answering chat reported. See `token_count`

    @property
    def cache(self): return self.rec.cache

    @property
    def token_count(self):
        "Tokens the conversation occupies, as the chat that answered reported them."
        return self._ctx

    def cancel(self):
        "Stop the live turn, if one was ever built. Nothing to stop while replaying."
        return self._chat.cancel() if self._chat is not None else False

    @property
    def cancelled(self): return self._chat is not None and self._chat.cancelled

    @property
    def chat(self):
        "The real `Chat`, built only when something actually has to be asked."
        if self._chat is None:
            self._chat = Chat(self.model, sp=self.sp, tools=list(self.tools),
                              messages=self.hist, **self.kw)
        return self._chat

    def _fold(self, u, ctx=None):
        """Add one reply's usage block to `use`, the way `UsageCallback` does for a live chat.

        A recording holds what the call actually cost, so a replay reports it. Without this a
        replayed turn looked free, and anything sized by what a call cost could not be tested.
        """
        u = u or {}
        self.use += UsageStats(
            u.get('prompt_tokens', 0), u.get('completion_tokens', 0), u.get('total_tokens', 0), 1,
            cached_tokens=u.get('cached_tokens', 0), model=u.get('model') or self.model,
            reasoning_tokens=u.get('reasoning_tokens', 0),
            cache_creation_tokens=u.get('cache_creation_tokens', 0), cost=u.get('cost', 0.0))
        # a turn's usage block sums its model steps, so it is what the turn cost and not what the
        # window holds. Occupancy is the answering chat's own count, recorded beside the reply
        if ctx: self._ctx = ctx
        elif t := u.get('total_tokens'): self._ctx = t   # recorded before `ctx` was stored
        return self.use

    def _key(self, kind, *args, hist=True):
        "What a reply is recorded under. `hist=False` is for asks that are stateless by definition."
        return self.rec.key(self.model, kind, self.sp, self.kw,
                            [getattr(t, '__name__', str(t)) for t in self.tools],
                            [canon_msg(m) for m in self.hist] if hist else [], *args)

    def _ask(self, kind, args, f, hist=True):
        "Replay this ask if it is recorded, else run `f()` and record it."
        n = len(self.hist) if hist else 0
        what = f'{kind} for {self.model} after {n} messages: {str(args[0])[:80]}'
        return self.rec(self._key(kind, *args, hist=hist), f, what)

    def __call__(self, prompt, **kw):
        "One turn, replayed if it has been asked before. `hist` ends up as the real turn left it."
        key = self._key('call', prompt, kw)
        hit = key in self.rec.cache
        what = f'call for {self.model} after {len(self.hist)} messages: {str(prompt)[:80]}'
        def _live():
            n = len(self.chat.hist)
            r = self.chat(prompt, **kw)
            return dict(resp=as_resp(r), hist=[dict(m) for m in self.chat.hist[n:]],
                        ctx=getattr(self.chat, 'token_count', 0))
        rec = self.rec(key, _live, what)
        self._fold(rec['resp'].get('usage'), rec.get('ctx'))
        # a stopped turn is a truncated one, and a recording would replay the truncation forever
        if not hit and self.cancelled: self.rec.forget(key)
        self.hist += [dict(m) for m in rec['hist']]
        # a replayed turn never reached the live chat, so hand it the history it missed
        if hit and self._chat is not None:
            self._chat.hist = list(self.hist)
            self._chat._recreate_conv()
        return Resp(rec['resp'])

    def oneshot(self, prompt, sp='', think=None, max_tokens=None):
        return self._ask('oneshot', [prompt, sp, think, max_tokens], hist=False,
                         f=lambda: self.chat.oneshot(prompt, sp, think=think, max_tokens=max_tokens))

    def classify(self, text, labels, **kw):
        return self._ask('classify', [text, list(labels), kw], hist=False,
                         f=lambda: self.chat.classify(text, labels, **kw))

    def reconfigure(self, sp=None, tools=None):
        "Change `sp` or `tools` for the asks that follow. Both are in the key, so a replay stays honest."
        if sp is not None: self.sp = sp
        if tools is not None: self.tools = L(tools)
        if self._chat is not None: self._chat.reconfigure(sp=sp, tools=tools)
        return self

    def close(self):
        if self._chat is not None: self._chat.close(); self._chat = None

In [ ]:
built = []

class _CountChat(Chat):
    "Counts how often a real chat was built and asked, so replays can be told from live calls."
    _runtime, ctx_limit, token_count = 'count', 8192, 0
    def __init__(self, model=None, **kw):
        built.append(model)
        self._setup(model, ChatOpts.create(kw.pop('opts', None), **kw))
    def _send(self, msg, **kw):
        self.hist.append(self.mk_msg(msg))
        self.turn_res = Resp({'role': 'assistant', 'content': f'reply to {msg}'})
        self.hist.append(dict(self.turn_res))
        return self.turn_res
    def _oneshot(self, prompt, sp='', think=None, max_tokens=None): return f'oneshot: {prompt}'

register_runtime(Runtime('count', _CountChat, ('count-',)))

In [ ]:
d = tempfile.mkdtemp()
c = CachedChat('count-1', path=d, record=True, runtime='count')
test_eq(resp_text(c('hello')), 'reply to hello')
test_eq(built, ['count-1'])                # built once, on the first miss

In [ ]:
built.clear()
c2 = CachedChat('count-1', path=d, record=False, runtime='count')
test_eq(resp_text(c2('hello')), 'reply to hello')
test_eq(built, [])                         # replayed, so no chat was ever built
test_eq([m['role'] for m in c2.hist], ['user', 'assistant'])   # ...and the history came back

In [ ]:
# the conversation so far is part of the key, so the same prompt twice is two recordings
test_fail(lambda: c2('hello'), contains='no recording')
test_eq(resp_text(c('hello')), 'reply to hello')   # the recorder can, and records the second turn
test_eq(resp_text(c2('hello')), 'reply to hello')  # ...so now the replayer can too

In [ ]:
c3 = CachedChat('count-1', path=d, record=False, sp='a different briefing', runtime='count')
test_fail(lambda: c3('hello'), contains='no recording')   # sp is part of the key

In [ ]:
built.clear()
c4 = CachedChat('count-1', path=d, record=True, runtime='count')
test_eq(c4.oneshot('what is 2+2?'), 'oneshot: what is 2+2?')
c4('hello')                                          # a turn that changes the history
test_eq(c4.oneshot('what is 2+2?'), 'oneshot: what is 2+2?')
test_eq(len([b for b in built]), 1)                  # one chat; the second oneshot replayed

In [ ]:
test_eq(c4.reconfigure(sp='new').sp, 'new')
test_eq(c4.cancelled, False)
c4.close()
test_eq(c4.cancel(), False)                # nothing live to stop

`use` folds usage from live and replayed replies into the same counter. Recorded usage allows compaction, budgets, and context-window checks to run during replay.

In [ ]:
class _PaidChat(_CountChat):
    "Answers with a usage block, the way a hosted backend does."
    _runtime = 'paid'
    def _send(self, msg, **kw):
        self.hist.append(self.mk_msg(msg))
        self.turn_res = Resp({'role': 'assistant', 'content': f'reply to {msg}',
                              'usage': {'prompt_tokens': 100, 'completion_tokens': 7,
                                        'total_tokens': 107, 'cost': 0.002, 'model': 'paid-1'}})
        self.hist.append(dict(self.turn_res))
        return self.turn_res

register_runtime(Runtime('paid', _PaidChat, ('paid-',)))
e = tempfile.mkdtemp()
live = CachedChat('paid-1', path=e, record=True, runtime='paid')
live('hello')
live.use

In [ ]:
test_eq((live.use.total_tokens, live.use.completion_tokens, live.use.n), (107, 7, 1))
test_eq(live.use.cost, 0.002)
test_eq(live.use.model, 'paid-1')

# and the replay reports the same, without building a chat
built.clear()
replayed = CachedChat('paid-1', path=e, record=False, runtime='paid')
replayed('hello')
test_eq(built, [])                                        # nothing was built, so nothing was asked
test_eq(replayed.use.total_tokens, live.use.total_tokens)
test_eq(replayed.use.cost, live.use.cost)

# occupancy, not billing volume: what the window holds after the turn, live or replayed
test_eq((live.token_count, replayed.token_count), (107, 107))

# a reply with no usage block folds a turn and no tokens, rather than raising
plain = CachedChat('count-1', path=d, record=False, runtime='count')
plain('hello')
test_eq((plain.use.total_tokens, plain.use.n), (0, 1))
test_eq(plain.token_count, 0)

`use` and `token_count` are different numbers, and a turn with a tool call is where they part.
`_finish_turn` sums every model step into the reply's usage block and keeps only the last step's
total as occupancy, so a recording has to carry both. Folding the usage block into `token_count`
reported the whole turn's billing as the window's contents, and grew with each extra step.

In [ ]:
def weather(city: str):
    "Look up the weather."
    return f'{city}: fine'

class _StepChat(ToolLoopMixin, Chat):
    "Two model steps, one tool call between them. `ToolLoopMixin._send` does the real totalling."
    _runtime, ctx_limit = 'step', 8192
    def __init__(self, model=None, **kw):
        self._n = 0
        self._setup(model, ChatOpts.create(kw.pop('opts', None), **kw))
    def _model_step(self, **kw):
        self._n += 1
        if self._n == 1: return Resp({
            'role': 'assistant', 'content': '',
            'usage': {'prompt_tokens': 100, 'completion_tokens': 10, 'total_tokens': 110},
            'tool_calls': [{'id': '1', 'type': 'function',
                            'function': {'name': 'weather', 'arguments': {'city': 'Madurai'}}}]})
        return Resp({'role': 'assistant', 'content': 'fine',
                     'usage': {'prompt_tokens': 150, 'completion_tokens': 6, 'total_tokens': 156}})

register_runtime(Runtime('step', _StepChat, ('step-',)))
f = tempfile.mkdtemp()
stepped = CachedChat('step-1', path=f, record=True, runtime='step', tools=[weather])
res = stepped('weather in Madurai?')
res['usage'], stepped.token_count

In [ ]:
test_eq(res['usage']['total_tokens'], 266)      # 110 + 156: what the two steps together cost
test_eq(stepped.use.total_tokens, 266)          # ...which is what `use` counts
test_eq(stepped.token_count, 156)               # ...and the window holds the last step, not the sum

# the replay reports both, from the same recording
again = CachedChat('step-1', path=f, record=False, runtime='step', tools=[weather])
again('weather in Madurai?')
test_eq((again.use.total_tokens, again.token_count), (266, 156))

In [ ]:
#| hide
del RUNTIMES['count'], RUNTIMES['step']

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()